<a href="https://colab.research.google.com/github/Lisura123/J26-IT-348/blob/component2-tinyml/component2-tinyml/notebooks/colab_operation_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Component 2: independent OPERATION evaluation (Colab)

Evaluates the **frozen** final model (`models/`) **once** on the OPERATION window, following the protocol fixed in `docs/OPERATION_EVAL.md` (`operation-eval-v1`).

- Nothing is fitted or changed: no V_ref or scaler refit, no training, no threshold change.
- Inference uses only `models/model_contract.json` (NumPy). `label` and `health_score` are removed before inference.
- **Run once.** The evaluation step only runs after you set `CONFIRM_FINAL_TEST = True` in Step 5, and it refuses to overwrite results.
- OPEN/SHORT outputs are ML predictions, **not** confirmed electrical faults. No relay isolation is implemented. Latency is host CPU, **not ESP32**.

All code comes from `src/tinyml/holdout.py` and `scripts/06_evaluate_operation.py`. This notebook only calls it.

Run the cells in order with **Shift + Enter**. Send screenshots after each step as before.

## Step 1: Get the latest repository code and install dependencies

In [1]:
# STEP 1: Get the latest component2-tinyml code and install dependencies
import os, sys, subprocess

REPO_URL = "https://github.com/Lisura123/J26-IT-348"
BRANCH = "component2-tinyml"
REPO_DIR = "/content/J26-IT-348"
COMPONENT_DIR = f"{REPO_DIR}/component2-tinyml"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
print("Commit:", subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"],
                                capture_output=True, text=True, check=True).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                f"{COMPONENT_DIR}/requirements.txt"], check=True)
import numpy as np, pandas as pd, sklearn
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__)

Commit: 288afc3 Created using Colab
Python 3.13.16 | numpy 2.1.3 | pandas 2.2.3 | scikit-learn 1.6.1


## Step 2: Import the modules and verify the frozen model (no data is read)

In [2]:
# STEP 2: Import modules and verify the committed model artifacts
import importlib.util
SRC_DIR = f"{COMPONENT_DIR}/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from tinyml.validate import file_sha256
from tinyml.train import check_dataset_sha256
from tinyml.holdout import PROTOCOL_VERSION, verify_frozen_model

MODEL_DIR = f"{COMPONENT_DIR}/models"
contract = verify_frozen_model(MODEL_DIR)          # pinned SHA-256 values; stops on any difference
print("Protocol:", PROTOCOL_VERSION)
print("Model   :", contract["model_version"], "| status", contract["status"],
      "| trained at", contract["trained_on"]["git_commit"])
print("Weights :", contract["weights_sha256"])
print("Frozen model verified.")

spec = importlib.util.spec_from_file_location("op_eval", f"{COMPONENT_DIR}/scripts/06_evaluate_operation.py")
op_eval = importlib.util.module_from_spec(spec); spec.loader.exec_module(op_eval)

Protocol: operation-eval-v1
Model   : c2-local-v1 | status TRAINED | trained at 9a641d5
Weights : 94e7ab9384a1906ec2cf0d3176b435e700f15cd99552fbc6c71519c648ecc3a7
Frozen model verified.


## Step 3: Upload the dataset
Click **Choose Files** and select `pv_fault_dataset.csv`. The file stays only in this Colab session.

In [3]:
# STEP 3: Upload pv_fault_dataset.csv
from google.colab import files
uploaded = files.upload()
csv_files = [n for n in uploaded if n.endswith(".csv")]
if not csv_files:
    raise SystemExit("No CSV uploaded. Run this cell again.")
DATA_PATH = os.path.abspath(csv_files[0])
print("Dataset file:", DATA_PATH, "|", os.path.getsize(DATA_PATH), "bytes")

Saving pv_fault_dataset.csv to pv_fault_dataset.csv
Dataset file: /content/pv_fault_dataset.csv | 957366 bytes


## Step 4: Check the dataset fingerprint (stops if it is not the validated file)

In [4]:
# STEP 4: SHA-256 check
sha = check_dataset_sha256(file_sha256(DATA_PATH))
print("SHA-256 OK:", sha)

SHA-256 OK: 5fbb93471c056c274cbf81e8c317934e379b2484013147e4c2cacfa93421595b


## Step 5: Confirm the one-time final test
The OPERATION window is the independent final test and is evaluated **once**. Only continue when the code commit in Step 1 is the approved one.

To confirm, change `False` to `True` in the cell below, then run it.

In [8]:
# STEP 5: Explicit confirmation (change False to True by hand)
CONFIRM_FINAL_TEST = True
print("Confirmed:", CONFIRM_FINAL_TEST)

Confirmed: True


## Step 6: Run the OPERATION evaluation (once)
Writes `operation_eval.json` and `operation_eval.md` to `/content/operation_eval/`. Run this cell **once**; it refuses to overwrite existing results.

In [9]:
# STEP 6: Independent OPERATION evaluation (frozen model, run once)
if not CONFIRM_FINAL_TEST:
    raise SystemExit("Set CONFIRM_FINAL_TEST = True in Step 5 first.")
SPLITS_PATH = f"{COMPONENT_DIR}/results/splits_runwise_v1.json"
OUT_DIR = "/content/operation_eval"
result, paths = op_eval.run(DATA_PATH, SPLITS_PATH, MODEL_DIR, OUT_DIR,
                            confirm_final_test=True, code_commit=COMMIT)
for p in paths:
    print("wrote", p)

FINAL_TEST_OPERATION: wrote /content/operation_eval/operation_eval.json and /content/operation_eval/operation_eval.md
wrote /content/operation_eval/operation_eval.json
wrote /content/operation_eval/operation_eval.md


## Step 7: Show the report

In [10]:
# STEP 7: Show the Markdown report
from IPython.display import Markdown, display
display(Markdown(open(paths[1], encoding="utf-8").read()))

# Independent OPERATION evaluation (final test)

Status `FINAL_TEST_OPERATION`, protocol `operation-eval-v1`, window `OPERATION`, created 2026-10-18T12:20:11Z.

Model `c2-local-v1`, weights SHA-256 `94e7ab9384a1906ec2cf0d3176b435e700f15cd99552fbc6c71519c648ecc3a7`, trained at `9a641d5`; evaluation code `288afc3`. Dataset SHA-256 `5fbb93471c056c274cbf81e8c317934e379b2484013147e4c2cacfa93421595b`, split `runwise-v1`. Environment {'python': '3.13.16', 'numpy': '2.1.3', 'pandas': '2.2.3', 'sklearn': '1.6.1'}.

Frozen model: no refit, no retraining, no threshold change. Inference uses the model contract only (NumPy); `label` and `health_score` are removed before inference.

## Coverage (quality gates)

3000 readings in 50 runs; 2940 classified, 60 skipped {'LOW_LIGHT': 60}.

| Class | Readings | Classified | Coverage | Correct | Coverage-adjusted recall |
|---|---|---|---|---|---|
| NORMAL | 600 | 600 | 1.000 | 600 | 1.000 |
| PARTIAL_SHADING | 600 | 600 | 1.000 | 600 | 1.000 |
| DUST | 600 | 540 | 0.900 | 479 | 0.798 |
| OPEN_CIRCUIT | 600 | 600 | 1.000 | 600 | 1.000 |
| SHORT_CIRCUIT | 600 | 600 | 1.000 | 600 | 1.000 |

Runs not fully classified: SITE-D-DUS-12 (0/60).

## Classification (classified readings)

Accuracy 0.979, macro-F1 0.978, balanced accuracy 0.977, n = 2940.

| Class | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| NORMAL | 0.910 | 1.000 | 0.953 | 600 |
| PARTIAL_SHADING | 0.997 | 1.000 | 0.998 | 600 |
| DUST | 1.000 | 0.887 | 0.940 | 540 |
| OPEN_CIRCUIT | 1.000 | 1.000 | 1.000 | 600 |
| SHORT_CIRCUIT | 1.000 | 1.000 | 1.000 | 600 |

Confusion matrix (rows = true, columns = predicted):

| true \ pred | NOR | PAR | DUS | OPE | SHO |
|---|---|---|---|---|---|
| NOR | 600 | 0 | 0 | 0 | 0 |
| PAR | 0 | 600 | 0 | 0 | 0 |
| DUS | 59 | 2 | 479 | 0 | 0 |
| OPE | 0 | 0 | 0 | 600 | 0 |
| SHO | 0 | 0 | 0 | 0 | 600 |

Run level: majority vote 0.980 over 49 runs; worst run `SITE-C-DUS-06` (0.083).

95% intervals (run-level bootstrap, stratified by true class, percentile interval, 1000 resamples, seed 42): accuracy [0.941, 0.999]; macro_f1 [0.937, 0.999]; OPEN_CIRCUIT_recall [1.000, 1.000]; SHORT_CIRCUIT_recall [1.000, 1.000].

Per site: SITE-A 1.000 (n=480), SITE-B 1.000 (n=480), SITE-C 0.917 (n=660), SITE-D 0.989 (n=360), SITE-E 0.997 (n=780), SITE-F 1.000 (n=180).

## OPEN_CIRCUIT and SHORT_CIRCUIT safety analysis

Every OPEN/SHORT output is an **ML prediction, not electrically confirmed** (confirmation rules are placeholders). Labels are dataset labels. No relay isolation is implemented.

### OPEN_CIRCUIT

- False OPEN_CIRCUIT predictions: 0 (0.00 per 1,000 classified non-OPEN_CIRCUIT readings); by true class none.
- Missed OPEN_CIRCUIT faults: 0 of 600 classified fault readings; 0 skipped; recall 1.000, coverage-adjusted recall 1.000.
- Electrical confirmation: NOT_ASSESSED: confirmation rules are placeholders (PHASE7_RESULTS.md decision 3); predictions are unconfirmed.

| Fault run | Site | Classified | Correct | Missed (as) | Skipped | First detection (s) |
|---|---|---|---|---|---|---|
| SITE-B-OPE-18 | SITE-B | 60 | 60 | 0  | 0 | 0 |
| SITE-C-OPE-02 | SITE-C | 60 | 60 | 0  | 0 | 0 |
| SITE-C-OPE-06 | SITE-C | 60 | 60 | 0  | 0 | 0 |
| SITE-C-OPE-08 | SITE-C | 60 | 60 | 0  | 0 | 0 |
| SITE-C-OPE-10 | SITE-C | 60 | 60 | 0  | 0 | 0 |
| SITE-C-OPE-20 | SITE-C | 60 | 60 | 0  | 0 | 0 |
| SITE-E-OPE-12 | SITE-E | 60 | 60 | 0  | 0 | 0 |
| SITE-E-OPE-14 | SITE-E | 60 | 60 | 0  | 0 | 0 |
| SITE-F-OPE-04 | SITE-F | 60 | 60 | 0  | 0 | 0 |
| SITE-F-OPE-16 | SITE-F | 60 | 60 | 0  | 0 | 0 |

### SHORT_CIRCUIT

- False SHORT_CIRCUIT predictions: 0 (0.00 per 1,000 classified non-SHORT_CIRCUIT readings); by true class none.
- Missed SHORT_CIRCUIT faults: 0 of 600 classified fault readings; 0 skipped; recall 1.000, coverage-adjusted recall 1.000.
- Electrical confirmation: NOT_ASSESSED: confirmation rules are placeholders (PHASE7_RESULTS.md decision 3); predictions are unconfirmed.

| Fault run | Site | Classified | Correct | Missed (as) | Skipped | First detection (s) |
|---|---|---|---|---|---|---|
| SITE-A-SHO-14 | SITE-A | 60 | 60 | 0  | 0 | 0 |
| SITE-B-SHO-16 | SITE-B | 60 | 60 | 0  | 0 | 0 |
| SITE-C-SHO-04 | SITE-C | 60 | 60 | 0  | 0 | 0 |
| SITE-C-SHO-08 | SITE-C | 60 | 60 | 0  | 0 | 0 |
| SITE-C-SHO-18 | SITE-C | 60 | 60 | 0  | 0 | 0 |
| SITE-E-SHO-02 | SITE-E | 60 | 60 | 0  | 0 | 0 |
| SITE-E-SHO-06 | SITE-E | 60 | 60 | 0  | 0 | 0 |
| SITE-E-SHO-10 | SITE-E | 60 | 60 | 0  | 0 | 0 |
| SITE-E-SHO-12 | SITE-E | 60 | 60 | 0  | 0 | 0 |
| SITE-E-SHO-20 | SITE-E | 60 | 60 | 0  | 0 | 0 |

## Host CPU latency (not ESP32)

Python/NumPy on the evaluation host. ESP32 flash, RAM and latency must be measured on the device and are not estimated from these numbers.
Rows: TRAINING rows (labels and health_score removed); OPERATION is not used for timing. Host: {'machine': 'x86_64', 'processor': 'x86_64', 'cpu_count': 2, 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39'}.

| Measurement | n | mean µs | median µs | p95 µs | p99 µs |
|---|---|---|---|---|---|
| forward_pass_single_reading | 1000 | 34.0 | 32.8 | 43.3 | 56.7 |
| pandas_reference_pipeline_single_reading | 200 | 13916.6 | 13546.0 | 15656.1 | 20830.9 |

Batched pipeline: 3000 readings in 25.6 ms (8.5 µs per reading).

Phase 7 context (TRAINING model selection, not comparable as a test result): {'cv_pooled_macro_f1': 0.981, 'loso_training_pooled_macro_f1': 0.98, 'note': 'TRAINING-window model-selection estimates, shown for context only'}.

## Caveats

- OPEN/SHORT outputs are ML predictions; electrical confirmation is not implemented.
- Relay isolation is not implemented; nothing is actuated.
- Labels are dataset labels; dataset provenance is UNRESOLVED (DATA_PROVENANCE.md).
- Latency is host CPU, not ESP32.
- Low-light and saturation thresholds are placeholders and were not tuned.


## Step 8: Download the results
Downloads `operation_eval.zip`. **Send it back for review.** Nothing is committed until it is reviewed and approved.

In [11]:
# STEP 8: Zip and download
import shutil
zip_path = shutil.make_archive("/content/operation_eval", "zip", OUT_DIR)
files.download(zip_path)
print("Downloaded", zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded /content/operation_eval.zip
